# 01 — Problem Formulation (Phase 1)

This notebook fixes the **tasks, cohorts, targets, prediction times, and split design**
BEFORE any data-dependent preprocessing or outcome-informed selection.

The final test set is reserved here and must not be touched until Phase 5.

## 1. Task definitions

### Regression (accepted file)
- **Target:** `int_rate` (pre-pricing).
- **Prediction time:** at application / before pricing.
- **Eligible inputs:** borrower credit profile, requested amount, term, purpose,
  geography, and other pre-application attributes.
- **Banned inputs:** `grade`, `sub_grade`, `installment` (rate-determining),
  all post-origination fields, and any derived flags/ratios from those fields.

### Classification (accepted file)
- **Target (3 classes):**
  - `0 = Good`: Fully Paid
  - `1 = At risk`: Late (16–30), Late (31–120), In Grace Period
  - `2 = Default`: Charged Off, Default
- **Dropped:** loans with `loan_status == 'Current'` (outcome unknown).
- **Prediction time:** origination (post-pricing is fine for this task).
- **Banned inputs:** all post-origination fields and their derivatives.

### Clustering
- **Unit/population:** individual funded-loan records in the accepted dataset, using the training cohort for development. Borrower identifiers are unavailable, so these are loan-profile segments, not verified unique-person groups.
- **Goal:** identify interpretable credit-profile segments for portfolio description, not predict repayment outcomes or make lending decisions.
- **Evaluation plan:** compare k-Means and Ward clustering using stability, silhouette and substantive interpretation; outcomes are excluded from fitting.
- State-level grouping in hypothesis testing is a separate descriptive cross-file analysis, not the clustering population.
- **Features (examples):** FICO, DTI, credit-history length, utilization.

## 2. Cohort and split design

- **Cohort (regression & classification):** funded loans in `accepted_2007_to_2018Q4.csv`.
- **Chronological split** by `issue_d` (calendar cutoffs, same-month records stay together):
  - Train: up to 2014-12
  - Validation: 2015-01 to 2016-12
  - Test: 2017-01 onward (held out)

Rationale: avoids temporal leakage; test period is a later portfolio, which matches
the intended real use case of predicting on new loans.

**Temporal interpretation and label availability.** This is a chronological holdout
with retrospective outcome labels from the supplied snapshot, not a historical
backtest. The files do not establish a verified common outcome-observation date
or per-loan dates when final labels became available. We do not infer that cutoff
from scheduled-payment dates, claim that all labels were observable in 2014/2016,
or estimate real-time deployment performance. Later-originated known-outcome loans
are a selected subset; excluding Current loans does not remove maturity bias.
Repeated persons cannot be ruled out because member_id is unavailable. Loan IDs
and exact duplicates are checked; matching amount/date/income is not proof of a duplicate.

**Test-set handling.** Test data are stored and may be deterministically parsed or
transformed with training-fitted preprocessing, but are excluded from development
statistics, hypothesis tests, feature audits, and plots. Earlier exploratory versions
inspected test-period distributions; this cannot be undone. The corrected run avoids
further inspection, and the eventual report must disclose that prior exposure.

In [1]:
import sys, json, logging
from pathlib import Path

import pandas as pd
import numpy as np

# Make src importable when the notebook is run from ./notebooks
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import LendingClubLoader
from src.preprocessing import REGRESSION_BANNED, CLASSIFICATION_BANNED, leakage_columns
from src.splits import ChronologicalSplitter

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

In [2]:
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
ACCEPTED = DATA_RAW / 'accepted_2007_to_2018Q4.csv'
REJECTED = DATA_RAW / 'rejected_2007_to_2018Q4.csv'

# For a first pass we only need a few key columns to confirm the schema.
# Full loading happens in Notebook 02.
PREVIEW_COLS = ['id', 'loan_amnt', 'term', 'int_rate', 'grade', 'sub_grade',
                'issue_d', 'loan_status', 'annual_inc', 'dti', 'fico_range_low']

loader = LendingClubLoader(ACCEPTED, REJECTED, usecols=PREVIEW_COLS, nrows=50_000)
df_preview = loader.load_accepted()
df_preview.shape

INFO | Loading accepted file: /Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados/data/raw/accepted_2007_to_2018Q4.csv


INFO | Accepted shape: (50000, 11)


(50000, 11)

## 3. Column eligibility

We record the eligibility decision for every column of interest.
The notebook writes this as `outputs/tables/column_eligibility.csv`
so it can be reused in the mid-journey report.

In [3]:
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'
OUT_TABLES.mkdir(parents=True, exist_ok=True)

eligibility = pd.DataFrame([
    # Regression task
    ('loan_amnt', 'regression', 'input', 'fixed at application'),
    ('term', 'regression', 'input', 'fixed at application'),
    ('purpose', 'regression', 'input', 'fixed at application'),
    ('annual_inc', 'regression', 'input', 'application data'),
    ('dti', 'regression', 'input', 'application data'),
    ('fico_range_low', 'regression', 'input', 'credit bureau at application'),
    ('grade', 'regression', 'EXCLUDE', 'rate-determining'),
    ('sub_grade', 'regression', 'EXCLUDE', 'rate-determining'),
    ('installment', 'regression', 'EXCLUDE', 'derived from loan_amnt/term/int_rate'),
    ('issue_d / credit_hist_months', 'regression', 'EXCLUDE', 'funding date is not available before pricing'),
    ('verification_status', 'regression', 'EXCLUDE', 'verification completion before pricing is not established'),
    ('funded_amnt / funded_amnt_inv', 'regression', 'EXCLUDE', 'funding occurs after pricing'),
    # Classification task
    ('loan_amnt', 'classification', 'input', 'origination snapshot'),
    ('term', 'classification', 'input', 'origination snapshot'),
    ('installment', 'classification', 'input', 'fixed at origination'),
    ('issue_d', 'classification', 'split-only', 'used for chronological split'),
    # Leakage (both tasks)
    ('total_pymnt', 'both', 'EXCLUDE', 'post-origination'),
    ('recoveries', 'both', 'EXCLUDE', 'post-origination'),
    ('last_fico_range_low', 'both', 'EXCLUDE', 'post-origination'),
    ('hardship_flag', 'both', 'EXCLUDE', 'post-origination prefix'),
    ('settlement_status', 'both', 'EXCLUDE', 'post-origination prefix'),
], columns=['column', 'task', 'decision', 'reason'])

eligibility.to_csv(OUT_TABLES / 'column_eligibility.csv', index=False)
eligibility

,column,task,decision,reason
0,loan_amnt,regression,input,fixed at application
1,term,regression,input,fixed at application
2,purpose,regression,input,fixed at application
3,annual_inc,regression,input,application data
4,dti,regression,input,application data
5,fico_range_low,regression,input,credit bureau at application
6,grade,regression,EXCLUDE,rate-determining
7,sub_grade,regression,EXCLUDE,rate-determining
8,installment,regression,EXCLUDE,derived from loan_amnt/term/int_rate
9,issue_d / credit_hist_months,regression,EXCLUDE,funding date is not available before pricing


## 4. Leakage guard lists

Two hard-coded lists (imported from `src.preprocessing`) are the single source of truth
for what must never become a predictor.

In [4]:
print('REGRESSION_BANNED (first 10):', REGRESSION_BANNED[:10])
print('CLASSIFICATION_BANNED (first 10):', CLASSIFICATION_BANNED[:10])
print('Leakage columns present in preview:', leakage_columns(df_preview))

REGRESSION_BANNED (first 10): ['collection_recovery_fee', 'credit_age_ratio', 'credit_hist_months', 'debt_settlement_flag', 'debt_settlement_flag_date', 'funded_amnt', 'funded_amnt_inv', 'grade', 'installment', 'int_rate']
CLASSIFICATION_BANNED (first 10): ['collection_recovery_fee', 'debt_settlement_flag', 'debt_settlement_flag_date', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'last_pymnt_amt', 'last_pymnt_d', 'loan_status', 'next_pymnt_d']
Leakage columns present in preview: []


## 5. Split definition

We freeze the split here. Notebook 02 will reuse this and write the split IDs to disk.

In [5]:
split_cfg = {
    'date_col': 'issue_d',
    'train_end': '2014-12',
    'val_end': '2016-12',
}

with open(PROJECT_ROOT / 'outputs' / 'split_definition.json', 'w') as f:
    json.dump(split_cfg, f, indent=2)

split_cfg

{'date_col': 'issue_d', 'train_end': '2014-12', 'val_end': '2016-12'}

## 6. Notes on assumptions and limitations

- **issue_d** is the funding month, not the application date. For the pre-pricing
  regression task we acknowledge this is an approximation and use it only as a
  chronological ordering variable, not as a predictor or a reference date for derived regression features. Baseline `mo_sin_old_rev_tl_op` may describe history age instead.
- **Joint/secondary-applicant fields** are not automatically leakage. We will
  distinguish non-applicability from missing information in Notebook 02 and will
  not replace absent secondary-applicant values with zero.
- **External temporal variables** must respect publication timing; we do not
  introduce any in this project.
- **Fairness:** geography and other sensitive proxies will be treated as a
  separate fairness consideration in the final report.
- **Verification:** verification_status is excluded from pre-pricing regression because completion timing is unverified. For the after-pricing classifier, the origination snapshot is assumed to include completed verification; this assumption needs validation before deployment.
- **Units:** amounts/income/balances are USD; int_rate, DTI and utilization are percentage points (13.5 means 13.5%, not 0.135); term is months; history age is months; employment length is years; bureau scores and account counts retain their original scales.